<a href="https://colab.research.google.com/github/Sasidhar015/Data-engineering-training-2026/blob/main/26_10_2026PySpark/PySpark_Practice_Assesments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install pyspark

In [2]:
from google.colab import files

uploaded = files.upload()

Saving lazy_execution_products.csv to lazy_execution_products.csv
Saving lazy_execution_sales_25000.csv to lazy_execution_sales_25000.csv
Saving lazy_execution_stores.csv to lazy_execution_stores.csv


In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, desc, sum, count

spark = SparkSession.builder \
    .appName("LazyExecutionPySpark") \
    .getOrCreate()


sales_df = spark.read.csv(
    "lazy_execution_sales_25000.csv",
    header=True,
    inferSchema=True)

products_df = spark.read.csv(
    "lazy_execution_products.csv",
    header=True,
    inferSchema=True)

stores_df = spark.read.csv(
    "lazy_execution_stores.csv",
    header=True,
    inferSchema=True)


sales_df.show(5)
products_df.show(5)
stores_df.show(5)
sales_df.printSchema()
products_df.printSchema()
stores_df.printSchema()

+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
| O000001|2026-06-22|     S04|      P002|       3|      2999|          10|   Net Banking|   Completed|     In Store|
| O000002|2026-05-29|     S07|      P014|       6|     29999|           0|          Card|   Completed|       Online|
| O000003|2026-06-27|     S06|      P008|       6|     22999|           0|          Card|   Completed|     In Store|
| O000004|2026-04-10|     S03|      P008|       7|     22999|          15|          Cash|   Completed|       Online|
| O000005|2026-03-14|     S06|      P007|       2|     79999|           0|          Cash|   Completed|       Online|
+--------+----------+--------+----------+--------+----------+---

In [29]:
q1 = sales_df.filter(col("quantity") > 5)

print("Transformation created successfully")

Transformation created successfully


In [28]:
q2 = q1.select(
    "order_id",
    "product_id",
    "quantity",
    "unit_price")

q2.show()

+--------+----------+--------+----------+
|order_id|product_id|quantity|unit_price|
+--------+----------+--------+----------+
| O000002|      P014|       6|     29999|
| O000003|      P008|       6|     22999|
| O000004|      P008|       7|     22999|
| O000006|      P005|       8|      2499|
| O000007|      P010|       8|      3999|
| O000008|      P002|       9|      2999|
| O000012|      P010|       9|      3999|
| O000013|      P013|       6|      5999|
| O000014|      P005|       8|      2499|
| O000015|      P007|       7|     79999|
| O000017|      P004|       9|     16999|
| O000018|      P012|       7|      6999|
| O000020|      P010|       7|      3999|
| O000021|      P006|       7|     55999|
| O000022|      P003|       7|      1899|
| O000023|      P010|       9|      3999|
| O000024|      P009|       7|      1999|
| O000028|      P014|       7|     29999|
| O000033|      P002|       6|      2999|
| O000034|      P012|       6|      6999|
+--------+----------+--------+----

In [6]:
q3 = sales_df.filter(col("unit_price") > 20000)

total_records = q3.count()

print("Total records:", total_records)

Total records: 6560


In [26]:
q3 = sales_df.filter(col("unit_price") > 20000)

total_records = q3.count()

print("Total records:", total_records)

Total records: 6560


In [27]:
q4 = sales_df.withColumn("gross_amount",col("quantity") * col("unit_price"))

q4.explain()

q4.show(10)

== Physical Plan ==
*(1) Project [order_id#17, order_date#18, store_id#19, product_id#20, quantity#21, unit_price#22, discount_pct#23, payment_method#24, order_status#25, sales_channel#26, (quantity#21 * unit_price#22) AS gross_amount#690]
+- FileScan csv [order_id#17,order_date#18,store_id#19,product_id#20,quantity#21,unit_price#22,discount_pct#23,payment_method#24,order_status#25,sales_channel#26] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/lazy_execution_sales_25000.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<order_id:string,order_date:date,store_id:string,product_id:string,quantity:int,unit_price:...


+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|gross_amount|
+--------+----------+--------+----------+--------+-

In [25]:
q5 = sales_df.filter(
    col("order_status") == "Completed").orderBy(
    desc("unit_price"))

records = q5.take(10)

for record in records:
    print(record)

Row(order_id='O000085', order_date=datetime.date(2026, 4, 22), store_id='S03', product_id='P007', quantity=4, unit_price=79999, discount_pct=0, payment_method='Card', order_status='Completed', sales_channel='Online')
Row(order_id='O000290', order_date=datetime.date(2026, 6, 26), store_id='S06', product_id='P007', quantity=3, unit_price=79999, discount_pct=10, payment_method='Net Banking', order_status='Completed', sales_channel='Online')
Row(order_id='O000098', order_date=datetime.date(2026, 4, 15), store_id='S05', product_id='P007', quantity=1, unit_price=79999, discount_pct=5, payment_method='Net Banking', order_status='Completed', sales_channel='Online')
Row(order_id='O000005', order_date=datetime.date(2026, 3, 14), store_id='S06', product_id='P007', quantity=2, unit_price=79999, discount_pct=0, payment_method='Cash', order_status='Completed', sales_channel='Online')
Row(order_id='O000107', order_date=datetime.date(2026, 5, 24), store_id='S05', product_id='P007', quantity=9, unit_pr

In [24]:
q6 = sales_df.filter(
    col("order_status") == "Completed").withColumn(
    "gross_amount",
    col("quantity") * col("unit_price")).select(
    "order_id",
    "product_id",
    "quantity",
    "unit_price",
    "gross_amount").orderBy(
    desc("gross_amount"))

print("All transformations created")
q6.show(20)

All transformations created
+--------+----------+--------+----------+------------+
|order_id|product_id|quantity|unit_price|gross_amount|
+--------+----------+--------+----------+------------+
| O002603|      P007|       9|     79999|      719991|
| O006170|      P007|       9|     79999|      719991|
| O002693|      P007|       9|     79999|      719991|
| O000107|      P007|       9|     79999|      719991|
| O002841|      P007|       9|     79999|      719991|
| O000367|      P007|       9|     79999|      719991|
| O002940|      P007|       9|     79999|      719991|
| O000527|      P007|       9|     79999|      719991|
| O003108|      P007|       9|     79999|      719991|
| O000690|      P007|       9|     79999|      719991|
| O003542|      P007|       9|     79999|      719991|
| O001201|      P007|       9|     79999|      719991|
| O003884|      P007|       9|     79999|      719991|
| O001300|      P007|       9|     79999|      719991|
| O003897|      P007|       9|     79

In [11]:
q7 = sales_df.filter(
    col("discount_pct") >= 10).withColumn(
    "discounted_sales_amount",
    col("quantity") * col("unit_price") *
    (1 - col("discount_pct") / 100)).orderBy(
    desc("discounted_sales_amount"))

q7.show(15)

+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+-----------------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|discounted_sales_amount|
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+-----------------------+
| O006653|2026-06-29|     S01|      P007|       9|     79999|          10|          Card|   Cancelled|       Online|               647991.9|
| O011893|2026-03-26|     S07|      P007|       9|     79999|          10|           UPI|   Completed|       Online|               647991.9|
| O006699|2026-06-25|     S05|      P007|       9|     79999|          10|   Net Banking|   Completed|       Online|               647991.9|
| O003108|2026-02-23|     S08|      P007|       9|     79999|          10|   Net Banking|   Completed|     In Store|               647991.9|
| O007743|202

In [12]:
q8 = sales_df.groupBy(
    "payment_method").agg(
    sum("quantity").alias("total_quantity_sold")).orderBy(
    desc("total_quantity_sold"))

q8.show()

+--------------+-------------------+
|payment_method|total_quantity_sold|
+--------------+-------------------+
|          Cash|              31754|
|          Card|              31327|
|           UPI|              30894|
|   Net Banking|              30889|
+--------------+-------------------+



In [13]:
q9 = sales_df.groupBy(
    "sales_channel",
    "order_status").agg(
    count("*").alias("transaction_count")).orderBy(
    "sales_channel",
    "order_status")
q9.show()

+-------------+------------+-----------------+
|sales_channel|order_status|transaction_count|
+-------------+------------+-----------------+
|     In Store|   Cancelled|              592|
|     In Store|   Completed|            10978|
|     In Store|    Returned|             1013|
|       Online|   Cancelled|              633|
|       Online|   Completed|            10797|
|       Online|    Returned|              987|
+-------------+------------+-----------------+



In [14]:
q10 = sales_df.select(
    "order_id",
    "store_id",
    "product_id",
    "quantity")

q10 = q10.drop("store_id")

print(q10.first())

Row(order_id='O000001', product_id='P002', quantity=3)


In [23]:
q11 = sales_df.join(
    products_df,
    on="product_id",
    how="inner").select(
    "product_name",
    "category",
    "quantity",
    "unit_price")

q11.show(20)

+-------------------+-----------+--------+----------+
|       product_name|   category|quantity|unit_price|
+-------------------+-----------+--------+----------+
|Mechanical Keyboard|Accessories|       3|      2999|
|      Smartphone 5G|    Mobiles|       6|     29999|
|     Android Tablet|    Tablets|       6|     22999|
|     Android Tablet|    Tablets|       7|     22999|
|          Laptop 16|  Computers|       2|     79999|
|          Webcam HD|Accessories|       8|      2499|
|  Bluetooth Speaker|      Audio|       8|      3999|
|Mechanical Keyboard|Accessories|       9|      2999|
|          Laptop 16|  Computers|       5|     79999|
|          Laptop 14|  Computers|       5|     55999|
|          Laptop 16|  Computers|       1|     79999|
|  Bluetooth Speaker|      Audio|       9|      3999|
|   External HDD 2TB|    Storage|       6|      5999|
|          Webcam HD|Accessories|       8|      2499|
|          Laptop 16|  Computers|       7|     79999|
|          Laptop 14|  Compu

In [16]:
q12 = sales_df.join(
    stores_df,
    on="store_id",
    how="inner").groupBy(
    "city").agg(
    sum("quantity").alias("total_quantity_sold")).orderBy(
    desc("total_quantity_sold"))
q12.show()

+---------+-------------------+
|     city|total_quantity_sold|
+---------+-------------------+
|   Jaipur|              16019|
|Hyderabad|              15784|
|  Chennai|              15727|
|Bengaluru|              15718|
|  Kolkata|              15619|
|   Mumbai|              15489|
|     Pune|              15407|
|    Delhi|              15101|
+---------+-------------------+



In [17]:
completed_sales = sales_df.filter(col("order_status") == "Completed")

q13 = completed_sales.join(
    products_df,
    on="product_id",
    how="inner").join(
    stores_df,
    on="store_id",
    how="inner").withColumn(
    "revenue",
    col("quantity") * col("unit_price")).groupBy(
    "category",
    "region").agg(
    sum("revenue").alias("total_completed_revenue")).orderBy(
    desc("total_completed_revenue"))

q13.show()

+-----------+------+-----------------------+
|   category|region|total_completed_revenue|
+-----------+------+-----------------------+
|  Computers| South|              371954527|
|  Computers| North|              240756425|
|  Computers|  West|              221092729|
|  Computers|  East|              122990192|
|    Mobiles| South|              120674524|
|    Mobiles|  West|               80372388|
|    Mobiles| North|               75482543|
|    Tablets| South|               64006217|
|   Displays| South|               45761308|
|    Mobiles|  East|               41178126|
|    Tablets| North|               38270336|
|    Tablets|  West|               37488370|
|    Storage| South|               36976319|
|      Audio| South|               34629665|
|   Displays| North|               32468090|
|   Displays|  West|               28286336|
|Accessories| South|               28139018|
|      Audio|  West|               25317111|
|      Audio| North|               24327211|
|    Stora

In [18]:
q14 = sales_df.filter(
    col("order_status") == "Completed").join(
    products_df,
    on="product_id",
    how="inner").withColumn(
    "discounted_revenue",
    col("quantity") * col("unit_price") *
    (1 - col("discount_pct") / 100)).groupBy(
    "product_id",
    "product_name").agg(
    sum("discounted_revenue").alias("total_discounted_revenue")).orderBy(
    desc("total_discounted_revenue"))
q14.show(5)

+----------+---------------+------------------------+
|product_id|   product_name|total_discounted_revenue|
+----------+---------------+------------------------+
|      P007|      Laptop 16|    5.2272546585000014E8|
|      P006|      Laptop 14|     3.830051604999999E8|
|      P014|  Smartphone 5G|     2.060406317500002E8|
|      P008| Android Tablet|    1.5040656029999995E8|
|      P004|27-inch Monitor|    1.1406074015000013E8|
+----------+---------------+------------------------+
only showing top 5 rows


In [19]:

step1 = sales_df.filter(col("quantity") > 5)

print("Transformation 1 completed: filter")

step2 = step1.select(
    "store_id",
    "product_id",
    "quantity",
    "unit_price")

print("Transformation 2 completed: select")

step3 = step2.withColumn(
    "gross_amount",
    col("quantity") * col("unit_price"))

print("Transformation 3 completed: withColumn")


step4 = step3.groupBy(
    "store_id").agg(
    sum("gross_amount").alias("total_gross_amount"))

print("Transformation 4 completed: groupBy")


step5 = step4.orderBy(
    desc("total_gross_amount"))

print("Transformation 5 completed: orderBy")

step5.show()

Transformation 1 completed: filter
Transformation 2 completed: select
Transformation 3 completed: withColumn
Transformation 4 completed: groupBy
Transformation 5 completed: orderBy
+--------+------------------+
|store_id|total_gross_amount|
+--------+------------------+
|     S07|         181665344|
|     S03|         179574561|
|     S02|         178450336|
|     S01|         173279127|
|     S08|         173063202|
|     S04|         167656967|
|     S05|         167165009|
|     S06|         162075553|
+--------+------------------+



In [20]:
q16 = sales_df.filter(
    col("unit_price") > 20000)

q16.explain()

q16.show(10)

print("Total records:", q16.count())

== Physical Plan ==
*(1) Filter (isnotnull(unit_price#22) AND (unit_price#22 > 20000))
+- FileScan csv [order_id#17,order_date#18,store_id#19,product_id#20,quantity#21,unit_price#22,discount_pct#23,payment_method#24,order_status#25,sales_channel#26] Batched: false, DataFilters: [isnotnull(unit_price#22), (unit_price#22 > 20000)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/lazy_execution_sales_25000.csv], PartitionFilters: [], PushedFilters: [IsNotNull(unit_price), GreaterThan(unit_price,20000)], ReadSchema: struct<order_id:string,order_date:date,store_id:string,product_id:string,quantity:int,unit_price:...


+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
| O000002|20

In [21]:
q17 = sales_df.join(
    products_df,
    on="product_id",
    how="inner").groupBy(
    "category").agg(
    sum("quantity").alias("total_quantity"))

records = q17.collect()

for record in records:
    print(record)

Row(category='Computers', total_quantity=16258)
Row(category='Displays', total_quantity=7870)
Row(category='Mobiles', total_quantity=16572)
Row(category='Accessories', total_quantity=41839)
Row(category='Storage', total_quantity=17251)
Row(category='Audio', total_quantity=17159)
Row(category='Tablets', total_quantity=7915)


In [22]:
q18 = sales_df.filter(
    col("order_status") == "Completed").withColumn(
    "discounted_revenue",
    col("quantity") * col("unit_price") *
    (1 - col("discount_pct") / 100))

q18.write.mode("overwrite").parquet("/content/completed_orders_discounted_revenue")

print("Parquet file saved successfully!")

Parquet file saved successfully!
